# WLASL Variant Data Hunt
Tries two alternative Kaggle WLASL uploads to find videos missing from our original download.
Reports how many new landmarks were added for our 50 target classes.

**Runtime:** T4 GPU recommended. Run cells top to bottom.

In [ ]:
# Cell 1 — Install dependencies
!pip install mediapipe kaggle --quiet

In [ ]:
# Cell 2 — Mount Drive and set up Kaggle credentials
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Copy kaggle.json from Drive (upload it to SignBridge/ on Drive if not there)
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/SignBridge/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials ready.')

In [ ]:
# Cell 3 — Clone repo and restore existing landmarks
import zipfile, glob

if not os.path.exists('/content/SignSync'):
    !git clone https://github.com/HaideyAli/SignSync.git /content/SignSync

os.makedirs('/content/SignSync/data/landmarks', exist_ok=True)
os.makedirs('/content/SignSync/data', exist_ok=True)

shutil.copy('/content/drive/MyDrive/SignBridge/data/labels_50.json',
            '/content/SignSync/data/labels_50.json')

zip_path = '/content/drive/MyDrive/SignBridge/data/landmarks.zip'
print('Restoring existing landmarks...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/SignSync/data/landmarks')

existing = set(os.path.basename(f) for f in glob.glob('/content/SignSync/data/landmarks/*.npy'))
print(f'Existing landmarks: {len(existing)}')

In [ ]:
# Cell 4 — Load our target labels and build the list of missing video IDs
import json
from pathlib import Path

with open('/content/SignSync/data/labels_50.json') as f:
    label_map = json.load(f)   # {word: idx}
TARGET_WORDS = set(label_map.keys())
print(f'Target words ({len(TARGET_WORDS)}): {sorted(TARGET_WORDS)}')

# Load nslt_100.json from Drive to know which video_ids map to our words
nslt_path = '/content/drive/MyDrive/SignBridge/data/nslt_100.json'
if not os.path.exists(nslt_path):
    # Try to copy from the SignSync data/raw folder if it was uploaded
    print('WARNING: nslt_100.json not found on Drive.')
    print('Upload data/raw/nslt_100.json and data/raw/wlasl_class_list.txt to Drive/SignBridge/data/')
else:
    with open(nslt_path) as f:
        nslt = json.load(f)

    # Load class list
    classlist_path = '/content/drive/MyDrive/SignBridge/data/wlasl_class_list.txt'
    idx_to_word = {}
    with open(classlist_path) as f:
        for line in f:
            line = line.strip()
            if line:
                idx, word = line.split('\t', 1)
                idx_to_word[int(idx)] = word

    # Find video_ids for our 50 words that we haven't extracted yet
    missing_videos = {}  # {video_id: word}
    for vid_id, info in nslt.items():
        word = idx_to_word.get(info['action'][0], '')
        if word in TARGET_WORDS:
            npy_name = f'{word}_{vid_id}.npy'
            if npy_name not in existing:
                missing_videos[vid_id] = word

    print(f'Missing video IDs for our 50 classes: {len(missing_videos)}')
    print('These are the videos we need to find in the alternative datasets.')

In [ ]:
# Cell 5 — MediaPipe setup
import urllib.request, mediapipe as mp, numpy as np, cv2
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

MODEL_PATH = Path('/content/holistic_landmarker.task')
MODEL_URL = ('https://storage.googleapis.com/mediapipe-models/'
             'holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task')
if not MODEL_PATH.exists():
    print('Downloading MediaPipe model...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

def landmarks_from_frame(result):
    def hand(lms):
        if lms: return np.array([[l.x,l.y,l.z] for l in lms], dtype=np.float32).flatten()
        return np.zeros(63, dtype=np.float32)
    def pose(lms):
        if lms: return np.array([[l.x,l.y,l.z,l.visibility or 0.] for l in lms], dtype=np.float32).flatten()
        return np.zeros(132, dtype=np.float32)
    return np.concatenate([hand(result.left_hand_landmarks),
                           hand(result.right_hand_landmarks),
                           pose(result.pose_landmarks)])

def process_video(video_path, detector):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.resize(frame, (640, 480))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        frames.append(landmarks_from_frame(detector.detect(mp_img)))
    cap.release()
    return np.array(frames, dtype=np.float32) if frames else np.zeros((1,258), dtype=np.float32)

opts = mp_vision.HolisticLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=mp_vision.RunningMode.IMAGE,
)
print('MediaPipe ready.')

In [ ]:
# Cell 6 — Try dataset 1: sttaseen/wlasl2000-resized
LANDMARKS_DIR = Path('/content/SignSync/data/landmarks')
VIDEOS_DIR = Path('/content/wlasl_extra/videos')

print('Downloading sttaseen/wlasl2000-resized...')
!kaggle datasets download -d sttaseen/wlasl2000-resized -p /content/wlasl_extra --unzip --quiet

# Auto-detect video files
video_files = list(Path('/content/wlasl_extra').rglob('*.mp4'))
print(f'Found {len(video_files)} .mp4 files')
if video_files:
    print(f'Example names: {[f.name for f in video_files[:5]]}')

In [ ]:
# Cell 7 — Extract landmarks from any new videos found in dataset 1
from tqdm import tqdm

# Build lookup: video_id -> video_path from downloaded files
id_to_path = {}
for vf in video_files:
    # Handle both '12345.mp4' and 'word_12345.mp4' naming styles
    stem = vf.stem
    vid_id = stem.split('_')[-1] if '_' in stem else stem
    id_to_path[vid_id] = vf

# Find intersection with our missing videos
found = {vid_id: word for vid_id, word in missing_videos.items() if vid_id in id_to_path}
print(f'New videos for our 50 classes in this dataset: {len(found)}')

if found:
    saved, errors = 0, 0
    with mp_vision.HolisticLandmarker.create_from_options(opts) as detector:
        for vid_id, word in tqdm(found.items(), desc='Extracting'):
            out_path = LANDMARKS_DIR / f'{word}_{vid_id}.npy'
            if out_path.exists():
                continue
            try:
                seq = process_video(id_to_path[vid_id], detector)
                np.save(out_path, seq)
                saved += 1
            except Exception as e:
                errors += 1
    print(f'Saved: {saved}  Errors: {errors}')
else:
    print('No new videos found — this dataset overlaps entirely with our existing download.')

# Free disk space before trying dataset 2
!rm -rf /content/wlasl_extra
print('Cleaned up dataset 1.')

In [ ]:
# Cell 8 — Try dataset 2: gazquez/wlasl-processed
print('Downloading gazquez/wlasl-processed...')
!kaggle datasets download -d gazquez/wlasl-processed -p /content/wlasl_extra2 --unzip --quiet

video_files2 = list(Path('/content/wlasl_extra2').rglob('*.mp4'))
print(f'Found {len(video_files2)} .mp4 files')
if video_files2:
    print(f'Example names: {[f.name for f in video_files2[:5]]}')

id_to_path2 = {}
for vf in video_files2:
    stem = vf.stem
    vid_id = stem.split('_')[-1] if '_' in stem else stem
    id_to_path2[vid_id] = vf

# Refresh missing list (may have reduced after dataset 1)
existing2 = set(f.name for f in LANDMARKS_DIR.glob('*.npy'))
still_missing = {vid_id: word for vid_id, word in missing_videos.items()
                 if f'{word}_{vid_id}.npy' not in existing2}

found2 = {vid_id: word for vid_id, word in still_missing.items() if vid_id in id_to_path2}
print(f'Additional new videos in dataset 2: {len(found2)}')

if found2:
    saved, errors = 0, 0
    with mp_vision.HolisticLandmarker.create_from_options(opts) as detector:
        for vid_id, word in tqdm(found2.items(), desc='Extracting'):
            out_path = LANDMARKS_DIR / f'{word}_{vid_id}.npy'
            if out_path.exists():
                continue
            try:
                seq = process_video(id_to_path2[vid_id], detector)
                np.save(out_path, seq)
                saved += 1
            except Exception as e:
                errors += 1
    print(f'Saved: {saved}  Errors: {errors}')
else:
    print('No additional new videos found in dataset 2 either.')

!rm -rf /content/wlasl_extra2

In [ ]:
# Cell 9 — Final report and save to Drive
from collections import Counter

final_npy = list(LANDMARKS_DIR.glob('*.npy'))
counts = Counter()
for f in final_npy:
    word = '_'.join(f.stem.split('_')[:-1])
    if word in TARGET_WORDS:
        counts[word] += 1

total = sum(counts.values())
print(f'Total landmarks for our 50 classes: {total} (was 598)')
print(f'Net new samples: {total - 598}')
print()
print(f'{"Word":<20} {"Samples":>8}')
print('-' * 30)
for word, c in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'{word:<20} {c:>8}')

# Save updated landmarks to Drive
if total > 598:
    print('\nSaving updated landmarks to Drive...')
    !cd /content/SignSync/data && zip -q -r /content/drive/MyDrive/SignBridge/data/landmarks.zip landmarks/
    print('Done. Use train_colab.ipynb to retrain.')
else:
    print('\nNo new samples found. landmarks.zip unchanged.')
    print('Recommendation: proceed to Option 4 (record your own signs).')